# What is a Phasor?

In [1]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import schemdraw
import schemdraw.elements as elm
from matplotlib import patches
from matplotlib.axes import Axes
from typing import Any

from src.drawing_utils import Arc, Arrow, Point, Segment, clear_axes
from src.figure_utils import process_figure
from src.plotting_utils import configure_matplotlib, format_xaxis, rm, set_discrete_colors
from src.schemdraw_utils import configure_schemdraw

configure_matplotlib()
configure_schemdraw()

## Phasors: A notation and way to make calculations straightforward

A phasor is analogous to a negative number, in the sense that both negative numbers and phasors are mathematical representations, notations, and ways to make calculations straightforward.

A negative number is a *mathematical representation* of the opposite of a quantity. It is distinct from the number zero which represents the absence or lack of a quantity. You can have no fewer than zero apples. Similarly, you cannot have a negative quantity of dollar bills. However, if the quantity of dollars refers not to physical legal tender but to an accounting of how many units of currency you have at your disposal, then a negative quantity represents the opposite of having dollars at your disposal&mdash;that is, owing dollars. In addition to this representation, negative numbers offer:

- A *notation*. By prefixing a regular number ($x$) with a minus sign ($-$), it becomes negative ($-x$).

- A *way to make calculations straightforward*. A negative number is like a different kind of quantity on which arithmetic operations can be done by treating them accordingly; to subtract a negative quantity you add its positive counterpart, to multiply you multiply by the positive counterpart and negate the product, and so on.

Similarly, a phasor is a *mathematical representation* of a sinusoid, in terms of its amplitude $A$ and phase $\theta$. A sinusoid already has a mathematical representation in the time domain with the notation $A \cos (\omega t + \theta)$. However, while this representation makes calculations possible, it does not make them straightforward and does not have a concise notation. Representing sinusoids in the phasor domain offers:

- A *notation*. Phasors can be written concisely, such as in polar form $A \angle \theta$. The "$\omega t$" is implicit.

- A *way to make calculations straightforward*. As we will see in [](3_life_with_and_without_phasors.ipynb), working in the phasor domain allows you to write and solve AC circuit analysis equations more quickly and easily than working in the time domain. Working in the time domain requires writing and solving differential equations from scratch for each circuit, even though the differential equations and their solutions are similar from one circuit to another. It turns out that the calculations that can be done with phasors are the bare minimum calculations that must be done in order to solve an AC circuit.

&nbsp;

## Back to basics: Treating sinusoids like vectors

In [ ]:
configure_matplotlib(fontsize=8)


def draw(
    ax: Axes,
    phi_X: float,
    theta: float = 0.0,
    angle_label_radius: float = 1.3,
    length: float = 3.0,
    text: Literal["all", "angle_only", None] = "angle_only",
    circle: bool = False,
    sine: bool = False,
    X: Point = Point(0, 0),
    suffix: str = "",
) -> Point:
    clear_axes(ax)

    X.drawn(ax)
    XY = (
        Segment.from_polar(origin=X, length=length, angle=phi_X)
        .rotated(theta)
        .drawn(ax)
    )
    XZ = Segment(start=X, end=Point(XY.end.x, X.y)).drawn(ax)
    Y = XY.end.drawn(ax)
    Z = XZ.end.drawn(ax)
    YZ = Segment(Y, Z).drawn(ax)

    if theta > 0:
        Segment.from_polar(
            origin=Point(0, 0), length=(XY.length + 0.2), angle=theta
        ).drawn(ax)
        theta_arc = Arc(vertex=X, start_angle=0, end_angle=theta).drawn(ax)
        Arrow(
            start=Segment.from_polar(origin=X, length=3, angle=(theta / 2)).end,
            end=theta_arc.mid,
        ).drawn(ax).start.labeled(ax, r"$\theta$", (0, -2.5), ha="left")

    phi_arc = (
        Arc(vertex=X, start_angle=0, end_angle=phi_X, label_radius=angle_label_radius)
        .rotated(theta)
        .drawn(ax)
    )
    if text in ["all", "angle_only"]:
        phi_arc.labeled(ax, rf"$\phi_X{suffix}$", (0, -2.5))

    if text == "all":
        X.labeled(ax, rf"$X{suffix}$", (5, 5))
        Y.labeled(ax, rf"$Y{suffix}$", (-7.5, 2.5))
        Z.labeled(ax, rf"$Z{suffix}$", (2.5, -10), ha="left")

        XY.mid.labeled(ax, rf"$X{suffix}Y{suffix}$", (-5, 5))
        YZ.mid.labeled(ax, rf"$Y{suffix}\!Z{suffix}$", (2.5, -5), ha="left")
        XZ.mid.labeled(ax, rf"$X{suffix}\!Z{suffix}$", (0, -12))

    if circle:
        Arc(
            vertex=Point(0, 0), start_angle=0, end_angle=phi_X, radius=XY.length
        ).rotated(theta).drawn(ax)
        XY.mid.labeled(ax, r"$A$", (-5, -7.5))

    if sine:
        ax.set_xlim((-6, 6))
        graph_origin = Point(0, 4)
        ax.set_ylim((-XY.length - 0.1, graph_origin.y + 2 * np.pi + 2))
        omega_t_axis = Arrow(
            Point(0, graph_origin.y - 0.2), Point(0, graph_origin.y + 2 * np.pi)
        ).drawn(ax)
        omega_t_axis.end.labeled(ax, r"$\phi_X \! = \! \omega t$", (0, 2.5))
        y_axis = Arrow(
            Point(-XY.length - 1.5, graph_origin.y),
            Point(XY.length + 1.5, graph_origin.y),
        ).drawn(ax)
        y_axis.end.labeled(ax, "$y$", (0, -1), ha="left")
        ax.plot(
            XY.length * np.cos(phi_arc.angles),
            graph_origin.y + phi_arc.angles,
            "k",
            linewidth=0.5,
        )
        ax.plot(
            [Y.x, XY.length * np.cos(theta + phi_X)],
            [Y.y, graph_origin.y + theta + phi_X],
            "k",
            linewidth=0.5,
            dashes=(5, 5),
        )

    return Y


fig, (ax1, ax2, ax3, ax4) = plt.subplots(
    4, figsize=(2, 7.75), layout="tight", gridspec_kw=dict(height_ratios=[1, 1.5, 3, 3])
)
fig.subplots_adjust(hspace=0.5)

draw(ax1, phi_X=(np.pi / 8), angle_label_radius=0.8, text="all")
draw(ax2, phi_X=(2 * np.pi - np.pi / 3), circle=True)
draw(ax3, phi_X=(2 * np.pi - np.pi / 3), circle=True, sine=True)
draw(ax4, theta=(np.pi / 6), phi_X=(2 * np.pi - np.pi / 3), circle=True, sine=True)

process_figure(fig, 2, 1)

```{figure} img/fig_2_1.png
:align: right
:figclass: margin
```

Consider the right triangle $XY\!Z$ in Figure 2.1(a). This triangle (like every triangle) has a ratio between length of side $X\!Z$ (adjacent to $\phi_X$) and the length of side $XY$ (the hypotenuse). The cosine function represents the relationship between this ratio and angle $\phi_X$:

$$
\cos \phi_X = \frac{X\!Z}{XY}
$$

As $\phi_X$ ranges from $0$ to $2 \pi$ radians (or $0$ to $360$ degrees), point $Y$ draws a circle centered around $X$ whose radius $A$ is the hypotenuse's length, as in Figure 2.1(b).

If we were to plot the length of side $X\!Z$ as a function of $\phi_X$, we would have a cosine wave of amplitude $A = XY$, as in Figure 2.1(c), that repeats every multiple of $2 \pi$ radians. This cosine wave may represent AC voltage or current.

$$
A \cos \phi_X
$$

If we wanted the drawing of our circle to depend on time $t$, and we wanted $f$ circles to be drawn per second, we would replace the independent variable $\phi_X$ with $t$:

$$
A \cos 2 \pi \! f t
$$

where $f$ is frequency in units of cycles per second (or *Hertz*, $\mathrm{Hz}$). For conciseness, we can replace $2 \pi \! f$ with $\omega$, the *natural frequency*, which is in units of radians per second:

$$
A \cos \omega t
$$

If we wanted to shift the drawing of our circle in time, we could add or subtract a constant from $t$. However, the mathematically equivalent convention is to add or subtract a constant $\theta$, called the *phase*, from $\omega t$. Now we're drawing our circle with a head start of $\theta$, from $\theta$ to $\theta + 2 \pi$ radians, as in Figure 2.1(d). This corresponds to a phase-shifted cosine wave:

$$
A \cos(\omega t + \theta)
$$

What if we wanted to add cosine waves? If the cosine wave represents AC voltage or current, being able to do so is important for AC circuit analysis. We can prove that the sum of two phase-shifted cosine waves is another phase-shifted cosine wave, and apply trigonometric identities to calculate it. However, the geometric approach is more intuitive. Consider that point $X$ of the first wave's triangle is at the origin of its two-dimensional space, and that the second cosine wave is similarly formed at any moment in time by a second triangle $X'Y'\!Z'$. If the two waves were added, then the second triangle would be translated such that its point $X'$ would be situated at the first triangle's point $Y$. Thus, the second triangle's point $Y'$ would have the coordinates $(X\!Z + X'\!Z', Y\!Z + Y'\!Z')$, representing the sum of the two waves. Similarly, if we were instead subtracting the second wave from the first, the point $Y'$ would have the coordinates $(X\!Z - X'\!Z', Y\!Z - Y'\!Z')$. Clearly, the addition and subtraction of the triangles by which cosine waves are formed satisfies the properties of vector addition and subtraction.

In [ ]:
fig, ax = plt.subplots(layout="tight")

Y = draw(ax, phi_X := np.pi / 3, angle_label_radius=0.5, text="all")
Yp = draw(
    ax, phi_Xp := np.pi / 8, angle_label_radius=0.525, text="all", X=Y, suffix="'"
)
draw(
    ax,
    Segment(Point(0, 0), Yp).angle,
    length=Segment(Point(0, 0), Yp).length,
    text=None,
)

configure_matplotlib()

process_figure(fig, 2, 2)

```{figure} img/fig_2_2.png
:align: center
:width: 64%
```

<p></p>

Phasors satisfy properties of vectors&mdash;addition, subtraction, multiplication by a scalar, and so on. However, phasors are not vectors. Vectors exist in $n$-dimensional space. As the geometric analogy suggested, phasors exist exclusively in two-dimensional space, and as we will learn in the next section, phasors must satisfy additional properties that vectors do not have.

## Reverse engineering the concept of phasors

Let's reverse-engineer the concept of phasors. Consider the following series RL circuit consisting of a voltage source $S$, a resistor $R$, and an inductor $L$:

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.6), layout="tight")
clear_axes(ax)

with schemdraw.Drawing(canvas=ax):
    v_s = elm.SourceSin().down(6).label(["$+$", "$v_S(t)$", "$-$"]).flip()
    elm.CurrentLabelInline(direction="out").at(v_s).label(r"$i(t)$", loc="bottom")
    elm.Line().right(5)
    R = elm.Resistor().up(3).label(["$-$", r"$v_\mathrm{R}(t)$  $R$", "$+$"], loc="bottom")
    L = (
        elm.Inductor()
        .up(3)
        .label(["$-$", r"$v_\mathrm{L}(t)$  $L$", "$+$"], loc="bottom")
        .flip()
    )
    elm.Line().left(5)

process_figure(fig, 2, 3)

```{figure} img/fig_2_3.png
:align: center
:width: 40%
```

<p></p>

The voltage source generates a cosine wave $v_S(t) = |V_{\! S}| \cos(\omega t + \theta_S)$. If using the geometric analogy to represent the voltage source's cosine wave, it would have coordinates $(V_{\! S x}, V_{\! S y}) = (|V_{\! S}| \cos \theta_S, |V_{\! S}| \sin \theta_S)$. Say that we want to solve for the steady-state current $i(t)$ through the circuit. We would start by writing the integro-differential equation:

$$
\begin{aligned}
i(t)
& = \frac{1}{L} \int_0^t v_L(\tau) \, \mathrm{d} \tau \\
& = \frac{1}{L} \int_0^t (v_S(\tau) - v_R(\tau)) \, \mathrm{d} \tau \\
& = \frac{1}{L} \int_0^t (v_S(\tau) - i(\tau) R) \, \mathrm{d} \tau \\
\end{aligned}
$$ (differential_eqn)

The steady-state solution to this integro-differential equation is as follows. Note that setting up and solving AC circuits' integro-differential equations will be discussed further in [Chapter 3](3_life_with_and_without_phasors.ipynb).

$$
i(t) = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \;\! \cos \! \left( \omega t + \theta_S - \arctan \frac{\omega L}{R} \right)
$$

Using the geometric analogy, we can represent the sinusoid of $i(t)$ using coordinates $(I_x, I_y)$. We can write $I_x$ as:

$$
\begin{aligned}
I_x
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \;\! \cos \! \left( \theta_S - \arctan \frac{\omega L}{R} \right) \\
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \left( \cos \theta_S \cdot \cos \! \left( \arctan \frac{\omega L}{R} \right) + \sin \theta_S \cdot \sin \! \left( \arctan \frac{\omega L}{R} \right) \right) \\
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \left( \cos \theta_S \cdot \frac{R}{\sqrt{R^2 + (\omega L)^2}} + \sin \theta_S \cdot \frac{\omega L}{\sqrt{R^2 + (\omega L)^2}} \right) \\
& = \frac{|V_{\! S}| \cos \theta_S \cdot R + |V_{\! S}| \sin \theta_S \cdot \omega L}{R^2 + (\omega L)^2}
\end{aligned}
$$ (I_x_deriv)

Similarly, $I_y$ can be written as follows.

$$
\begin{aligned}
I_y
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \;\! \sin \! \left( \theta_S - \arctan \frac{\omega L}{R} \right) \\
% & = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \left( \sin \theta_S \cos \arctan \frac{\omega L}{R} - \cos \theta_S \sin \arctan \frac{\omega L}{R} \right) \\
% & = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \left( \sin \theta_S \frac{R}{\sqrt{R^2 + (\omega L)^2}} - \cos \theta_S \frac{\omega L}{\sqrt{R^2 + (\omega L)^2}} \right) \\
& = \frac{|V_{\! S}| \sin \theta_S \cdot R - |V_{\! S}| \cos \theta_S \cdot \omega L}{R^2 + (\omega L)^2}
\end{aligned}
$$ (I_y_deriv)

Let's substitute $V_{\! S x}$ for $|V_{\! S}| \cos \theta_S$ and $V_{\! S y}$ for $|V_{\! S}| \sin \theta_S$. Let's also replace $R$ and $\omega L$ with new quantities $Z_x$ and $Z_y$, whose significance will become apparent shortly.

$$
(I_x, I_y) = \left( \frac{V_{\! S x} Z_x + V_{\! S y} Z_y}{Z_x^2 + Z_y^2}, \frac{V_{\! S y} Z_x - V_{\! S x} Z_y}{Z_x^2 + Z_y^2} \right)
$$

Making use of the properties of complex numbers (in the box below), we discover what is important about being able to write $(I_x, I_y)$ in this way. The complex number $I_x + {j} \, I_y$ (where ${j}$ is the imaginary unit) is the result of the following complex number division:

$$
I_x + {j} \, I_y = \frac{V_{\! S x} + {j} \, V_{\! S y}}{Z_x + {j} \, Z_y}
$$ (I_x_plus_j_I_y)

````{div} full-width
```{admonition} Properties of complex numbers
Addition and subtraction:

$$
(a + {j} \, b) \pm (c + {j} \, d) = (a \pm c) + {j} \, (b \pm d)
$$

Multiplication:

$$
(a + {j} \, b) (c + {j} \, d) = a c + {j} \, d a + {j} \, b c + ({j} \, b ) ({j} \, d) = (ac - bd) + {j} \, (bc + ad)
$$

Division:

$$
\begin{aligned}
\frac{a + {j} \, b}{c + {j} \, d}
& = \frac{a + {j} \, b}{c + {j} \, d} \frac{c - {j} \, d}{c - {j} \, d}
\quad \text{(Multiply both numerator and denominator by denominator's conjugate.)} \\
& = \frac{a c - {j} \, d a + {j} \, b c - ({j} \, b ) ({j} \, d)}{c^2 - {j} \, d c + {j} \, d c - ({j} \, d ) ({j} \, d)} \\
& = \frac{(ac + bd) + {j} \, (bc - ad)}{c^2 + d^2}
\end{aligned}
$$
```
````

Knowing that $Z_x = R$, Equation {eq}`I_x_plus_j_I_y` feels like Ohm's law. If we were to remove the inductor $L$ from the circuit and thus set $Z_y = \omega L$ to zero, the equation indeed becomes Ohm's law for a purely resistive circuit:

$$
\begin{aligned}
& I_x + {j} \, I_y = \frac{V_{\! S x} + {j} \, V_{\! S y}}{R} = \frac{|V_{\! S}| \cos \theta_S}{R} + {j} \, \frac{|V_{\! S}| \sin \theta_S}{R} \\
& \implies (I_x, I_y) = \left( \frac{|V_{\! S}| \cos \theta_S}{R}, \frac{|V_{\! S}| \sin \theta_S}{R} \right) \\
& \implies i(t) = \frac{v_S(t)}{R}
\end{aligned}
$$

In fact, Equation {eq}`I_x_plus_j_I_y` is the generalization of Ohm's law for an AC circuit with inductive and capacitive components, and $Z_x + {j} \, Z_y$, called *impedance*, is the generalization of electrical resistance $R$ for such a circuit. Just as we could write $i(t) = v_S(t) / R$ for a purely resistive AC circuit, we can write $I_x + {j} \, I_y = (V_{\! S x} + {j} \, V_{\! S y}) / (Z_x + {j} \, Z_y)$ for an AC circuit with inductive and capacitive components.

This indicates to us that the coordinates we've been working with in our geometric analogy are actually complex numbers. They satisfy all the properties of complex numbers, including addition, subtraction, multiplication by a scalar, multiplication, and division (as we've just seen). A complex number, as used to represent a sinusoidal voltage or current, is what we call a phasor. Furthermore, a complex number, as used to represent the ratio between a voltage phasor and a current phasor, is what we call impedance. Impedance, however, is not itself a phasor because it does not represent a sine wave.

The most important feature of phasors is that they allow us to do calculations using complex numbers rather than solving differential equations. For example, rather than solving differential equation {eq}`differential_eqn`, we can quickly convert $v_S(t)$ to a phasor, solve a regular equation in the phasor domain, and convert the solution to $i(t)$ back in the time domain.

You may be thinking that complex numbers are an unintuitive or downright far-fetched way to represent solving differential equations. In answer to this, consider that the behavior of multiplying a complex number of unit magnitude by ${j}$ is identical to the behavior of differentiating a sinusoid of unit amplitude with respect to time, as shown in the following diagram. Similarly, by reversing the diagram, the behavior of dividing a complex number of unit magnitude by ${j}$ is identical to the behavior of integrating a sinusoid of unit amplitude with respect to time. So, instead of writing derivatives and integrals to model capacitors and inductors, we can simply multiply and divide by ${j}$.

$$
\begin{array}{cccc}
& 1 & \cos t & \\
\text{Multiply by }{j} & \downarrow & \downarrow & \text{Differentiate w.r.t. } t \\
& {j} & - \sin t & \\
\text{Multiply by }{j} & \downarrow & \downarrow & \text{Differentiate w.r.t. } t \\
& -1 & - \cos t & \\
\text{Multiply by }{j} & \downarrow & \downarrow & \text{Differentiate w.r.t. } t \\
& -{j} & \sin t & \\
\text{Multiply by }{j} & \downarrow & \downarrow & \text{Differentiate w.r.t. } t \\
& 1 & \cos t &
\end{array}
$$

Notice too how there is a $+ \pi / 2$ (or $+90^\circ$) phase shift from $\cos t$ to $- \sin t$, and from $- \sin t$ to $- \cos t$, and so on. Thus, multiplying or dividing by ${j}$ in the phasor domain is also equivalent to shifting the corresponding sinusoid in the time domain by a quarter cycle.

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, layout="tight")

arrow_kw: dict[str, Any] = dict(
    color="k",
    linewidth=0.25,
    connectionstyle="arc3,rad=0.5",
    arrowstyle="simple,head_width=4,head_length=8",
)

ax1.set_aspect("equal")
set_discrete_colors(ax1)
ax1.plot([0, 1], [0, 0], ".-")
ax1.plot([0, 0], [0, 1], ".-")
ax1.plot([0, -1], [0, 0], ".-")
ax1.plot([0, 0], [0, -1], ".-")
ax1.plot([0], [0], "k.")
ax1.annotate("$1$", (1, 0), textcoords="offset points", xytext=(0, 7.5), ha="center")
ax1.annotate(
    "${j}$", (0, 1), textcoords="offset points", xytext=(-7.5, -2.5), ha="center"
)
ax1.annotate(
    "$-1$", (-1, 0), textcoords="offset points", xytext=(2.5, -12.5), ha="center"
)
ax1.annotate(
    "$-{j}$", (0, -1), textcoords="offset points", xytext=(12.5, -2.5), ha="center"
)
ax1.add_patch(patches.FancyArrowPatch((0.5, 0), (0, 0.5), **arrow_kw))
ax1.add_patch(patches.FancyArrowPatch((0, 0.5), (-0.5, 0), **arrow_kw))
ax1.add_patch(patches.FancyArrowPatch((-0.5, 0), (0, -0.5), **arrow_kw))
ax1.add_patch(patches.FancyArrowPatch((0, -0.5), (0.5, 0), **arrow_kw))
for i, xy in enumerate([(0.45, 0.55), (-0.45, 0.55), (-0.45, -0.55), (0.45, -0.55)]):
    ax1.annotate(
        f"{i + 1}. " + r"$\times {j}$",
        xy,
        textcoords="offset points",
        xytext=(0, -2.5),
        ha="center",
    )
ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_xlabel(r"$\mathrm{Re}$")
ax1.set_ylabel(r"$\mathrm{Im}$")
ax1.set_title(rm("Phasor domain (complex plane)"))

ts = np.linspace(0, 2 * np.pi)
set_discrete_colors(ax2)
ax2.plot(ts, np.cos(ts), label=r"$\cos t$")
ax2.plot(ts, -np.sin(ts), label=r"$- \sin t$")
ax2.plot(ts, -np.cos(ts), label=r"$- \cos t$")
ax2.plot(ts, np.sin(ts), label=r"$\sin t$")
format_xaxis(ax2, x_texts=True, start_at=0)
ax2.set_xlabel("$t$")
ax2.set_ylim([-1.5, 3])
ax2.set_yticks([-1, 0, 1])
ax2.add_patch(patches.FancyArrowPatch((2 * np.pi, 1), (3 * np.pi / 2, 1), **arrow_kw))
ax2.add_patch(patches.FancyArrowPatch((3 * np.pi / 2, 1), (np.pi, 1), **arrow_kw))
ax2.add_patch(patches.FancyArrowPatch((np.pi, 1), (np.pi / 2, 1), **arrow_kw))
ax2.add_patch(patches.FancyArrowPatch((np.pi / 2, 1), (0, 1), **arrow_kw))
for i, x in enumerate([7 * np.pi / 4, 5 * np.pi / 4, 3 * np.pi / 4, np.pi / 4]):
    ax2.annotate(
        f"{i + 1}.",
        (x, 1),
        textcoords="offset points",
        xytext=(0, 40),
        ha="center",
    )
    ax2.annotate(
        r"$\mathrm{d} / \mathrm{d} t$",
        (x, 1),
        textcoords="offset points",
        xytext=(0, 27.5),
        ha="center",
    )
    ax2.annotate(
        r"$+90^\circ$",
        (x, 1),
        textcoords="offset points",
        xytext=(0, 15),
        ha="center",
    )
ax2.legend(loc="lower right", bbox_to_anchor=(1.03, 1.125), ncols=2)
ax2.set_title(rm("Time domain"))

process_figure(fig, 2, 4, label_subfigures="no")

```{figure} img/fig_2_4.png
:align: center
:width: 64%
```

<p></p>

## Notation for phasors

Previously, we treated phasors as coordinates in two-dimensional space, with $x$ and $y$ components. For example, we had:

$$
(V_{\! S x}, V_{\! S y}) \qquad (I_x, I_y)
$$

However, now that we know that phasors (and impedances) are just complex numbers, we can replace this notation with complex number notation, with a real and imaginary component. For example:

$$
V_{\! S} = \mathrm{Re}(V_{\! S}) + {j} \, \mathrm{Im}(V_{\! S}) \qquad I = \mathrm{Re}(I) + {j} \, \mathrm{Im}(I) \qquad Z = \mathrm{Re}(Z) + {j} \, \mathrm{Im}(Z)
$$

### Polar form

Let's introduce a polar form of complex number notation, which is used to write a phasor (or impedance) in terms of amplitude and phase, rather than in terms of its real and imaginary components. For example, using Euler's formula $\mathrm{e}^{{j} x} = \cos x + {j} \sin x$, we can write $V_{\! S} = |V_{\! S}| \cos \theta_S + {j} \, |V_{\! S}| \sin \theta_S$ as:

$$
|V_{\! S}| \, \mathrm{e}^{{j} \theta_S}
$$

Or using *Steinmetz notation*:

$$
|V_{\! S}| \angle \theta_S
$$

We can just as well apply this polar form to equations {eq}`I_x_deriv` and {eq}`I_y_deriv`:

$$
\begin{aligned}
I
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \;\! \cos \! \left( \theta_S - \arctan \frac{\omega L}{R} \right) + {j} \, \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \;\! \sin \! \left( \theta_S - \arctan \frac{\omega L}{R} \right) \\
& = \frac{|V_{\! S}|}{\sqrt{R^2 + (\omega L)^2}} \: \angle \left( \theta_S - \arctan \frac{\omega L}{R} \right)
\end{aligned}
$$ (I_polar)

Previously, equations {eq}`I_x_deriv` and {eq}`I_y_deriv` led us to see a way that $I$ can be calculated from $V_{\! S}$ by dividing by impedance $Z$ using division of complex numbers. If we take a close look at Equation {eq}`I_polar`, we can find an additional, equivalent way in which $I$ can be calculated from $V_{\! S}$. Notice how $V_{\! S}$ can be converted to $I$ by dividing its amplitude $|V_{\! S}|$ by $\sqrt{R^2 + (\omega L)^2}$ and by subtracting $\arctan(\omega L / R)$ from $\theta_S$. This demonstrates a useful property of phasors and complex numbers in general in polar form&mdash;two complex numbers can easily be divided by dividing their amplitudes and subtracting their phases:

$$
\frac{A_1 \angle \theta_1}{A_2 \angle \theta_2} = \frac{A_1}{A_2} \angle (\theta_1 - \theta_2)
$$

Of course, all properties of complex numbers can be written in polar form, but multiplying and dividing complex numbers is easiest when they are in polar form. On the other hand, adding and subtracting complex numbers is easiest when they are in the non-polar form.

```{admonition} Properties of complex numbers (polar form)
Addition and subtraction:

$$
A_1 \angle \theta_1 \pm A_2 \angle \theta_2 = \sqrt{(A_1 \cos \theta_1 \pm A_2 \cos \theta_2)^2 + (A_1 \sin \theta_1 \pm A_2 \sin \theta_2)^2} \: \angle \arctan \frac{A_1 \sin \theta_1 \pm A_2 \sin \theta_2}{A_1 \cos \theta_1 \pm A_2 \cos \theta_2}
$$

Multiplication:

$$
A_1 \angle \theta_1 \cdot A_2 \angle \theta_2 = A_1 A_2 \angle (\theta_1 + \theta_2)
$$

Division:

$$
\frac{A_1 \angle \theta_1}{A_2 \angle \theta_2} = \frac{A_1}{A_2} \angle (\theta_1 - \theta_2)
$$
```

Complex numbers can be converted to and from polar form using the following formulae:

$$
\begin{gathered}
a + {j} \, b = \sqrt{a^2 + b^2} \: \angle \arctan \frac{b}{a} \\
A \angle \theta = A \cos \theta + {j} \, A \sin \theta
\end{gathered}
$$

## The phasor transformation

Thus far, we've looked at how phasors fit into AC circuit mathematics to see where they come from. At this point we can provide a formal definition for a phasor and how it is obtained from a sinusoid in the time domain. Consider the sinusoid $A \cos(\omega t + \theta)$. Adding an imaginary component ${j} \, A \sin(\omega t + \theta)$ gives us:

$$
A \cos(\omega t + \theta) + {j} \, \sin(\omega t + \theta)
$$

By setting $t = 0$ and thus removing the time-varying part $\omega t$, we can convert this to a phasor as follows. Setting $t$ to a specific value allows us to produce a "snapshot" of the sinusoid. Any specific value for $t$ will do, however $t = 0$ is most convenient.

$$
A \cos \theta + {j} \, \sin \theta
$$

If we convert to polar form,

$$
A \mathrm{e}^{{j} (\omega t + \theta)} = A \mathrm{e}^{{j} \omega t} \mathrm{e}^{{j} \theta},
$$

we can accomplish the same thing by dividing by the time-varying factor $\mathrm{e}^{{j} \omega t}$ to obtain:

$$
A \mathrm{e}^{{j} \theta}
$$